# EDA and data checks

What every input file contains, whether it is complete and consistent, what little cleaning it needs, and the first
hypotheses about what drives the ON/OFF decision. Everything later in `notebooks/` builds on the loader checked here
(`load` in `src/train_xgb_baseline.py`).

**Problem.** One scheduling *case* = one week (168 hours) starting at 00:00 UTC on day D. There is one case per day
(rolling horizon), so neighbouring cases share 6/7 of their hours. For each case the optimiser decided, for each of the
14 generating units and each hour, whether the unit is ON. We predict that probability. Metric: micro-average ROC-AUC.

**Data dictionary** (from `Dataset_definitions_and_explanation.pdf`)

| file | grain | columns | unit | meaning |
|---|---|---|---|---|
| `kernel/Historical_day_ahead_price_2015_2025` | hourly, UTC | 1 | EUR/MWh | market price for electricity sold in that hour; can be negative |
| `kernel/Historical_inflow_1958_2025` | hourly, UTC | 17 reservoirs + 1 river | m³/s | natural inflow into each reservoir |
| `kernel/Historical_volume_2015_2024` | daily, start of day | 17 reservoirs | Mm³ | stored water at 00:00 |
| `kernel/Synthetic_water_value_2015_2024` | daily | 17 reservoirs | EUR/MWh | marginal value of keeping one MWh of water in the reservoir |
| `kernel/Unit_commitment_decisions` | one row per case | 14 units × 168 h | 0/1 | the targets, plus `starttime` and `total_calculation_time` |
| `extended/Constraint_min_flow` | hourly, UTC | 6 rivers | m³/s | minimum river flow to be released; 0 = no requirement |
| `extended/Constraint_min_volume` | hourly, UTC | 3 reservoirs | Mm³ | minimum stored volume; 0 = no requirement |
| `extended/Tokke_Vinje_topology.yaml` | static | | | which reservoir feeds which plant, max volumes |

**Inputs the optimiser sees for case D** (and therefore the inputs we may use, for train and test alike): prices, inflows and
constraints for hours D 00:00 … D+6 23:00; volume and water value at D (initial state) and water value at D+7 (end-of-horizon
value of the water). `total_calculation_time` is an output of the optimiser and does not exist for test cases.

## 0. Setup

In [ ]:
import os, sys, re, glob
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from train_xgb_baseline import load, FeatureBuilder, H
from features_margin import add_margin_features, UPSTREAM

BLUE, INK, MUTED = "#2a78d6", "#0b0b0b", "#52514e"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED})
pd.set_option("display.width", 160)

KERNEL, EXTENDED = "data/kernel", "data/extended"
read = lambda path: pd.read_csv(path, parse_dates=["date"]).set_index("date")
raw = {os.path.basename(p)[:-4]: pd.read_csv(p) for p in sorted(glob.glob(f"{KERNEL}/*.csv") + glob.glob(f"{EXTENDED}/*.csv"))}
list(raw)

## 1. Raw file inventory

Shape, time range, duplicates and missing values of every CSV as delivered, before any processing.

In [ ]:
rows = []
for name, d in raw.items():
    tcol = "starttime" if "starttime" in d.columns else "date"
    t = pd.to_datetime(d[tcol], utc=True)
    rows.append({"file": name, "rows": len(d), "cols": d.shape[1] - 1, "first": t.min().date(), "last": t.max().date(),
                 "dup timestamps": int(t.duplicated().sum()), "NaN": int(d.isna().sum().sum()),
                 "typical step": t.diff().mode().iloc[0]})
pd.DataFrame(rows).set_index("file")

Nothing is missing and no timestamp is duplicated. Two things to handle in loading:

* **Price** is hourly but the file ends with 15-minute stamps (the last timestamp ends on `:45`). Where does the switch happen?
* **Inflow** starts in 1958; only 2015 onward is needed.

In [ ]:
p = read(f"{KERNEL}/Historical_day_ahead_price_2015_2025.csv")
steps = p.index.to_series().diff().value_counts()
first_15 = p.index[p.index.to_series().diff() == pd.Timedelta("15min")][0]
print("price time steps:", steps.to_dict())
print("first 15-minute step:", first_15, "-> after the last test case week (2024-12-25 + 7 d), so the test windows are hourly")
for n in ["Historical_volume_2015_2024", "Synthetic_water_value_2015_2024"]:
    d = read(f"{KERNEL}/{n}.csv"); print(f"{n}: steps {d.index.to_series().diff().value_counts().to_dict()}")

## 2. Value ranges per file (2015 to 2024)

In [ ]:
def ranges(d):
    d = d[(d.index >= "2015-01-01") & (d.index < "2025-01-01")]
    return d.agg(["min", "median", "mean", "max"]).T.round(2)

tables = {}
for f, unit in [("kernel/Historical_day_ahead_price_2015_2025", "EUR/MWh"), ("kernel/Historical_inflow_1958_2025", "m3/s"),
                ("kernel/Historical_volume_2015_2024", "Mm3"), ("kernel/Synthetic_water_value_2015_2024", "EUR/MWh"),
                ("extended/Constraint_min_volume", "Mm3"), ("extended/Constraint_min_flow", "m3/s")]:
    tables[f"{f.split('/')[-1]} [{unit}]"] = ranges(read(f"data/{f}.csv"))
pd.concat(tables)

Observations:

* Prices range from −62 to 898 EUR/MWh; the median (37) is far below the mean (58), so the distribution has a long right tail (2022).
* Inflow scales differ by two orders of magnitude between reservoirs (Vatjern median 0.08 m³/s, Songavatn 10 m³/s). Trees do not care, but any reservoir-level feature must be relative to the reservoir's own capacity.
* **Hyljelihyl and Vatjern volumes are constant** (min = max): the data description says these two are a fixed 50 % placeholder. They carry no information but are harmless to keep.
* Minimum-volume constraints are 0 (inactive) more than half the time; minimum flows are always at least a small positive value for five of six rivers.

## 3. Coverage: does every train and test case have its inputs?

In [ ]:
uc_raw = raw["Unit_commitment_decisions"]; test_map = pd.read_csv("prediction_mapping.csv", parse_dates=["starttime"])
sample = pd.read_csv("sample_submission.csv", nrows=1)
tr, te = pd.to_datetime(uc_raw["starttime"]), test_map["starttime"]
label_cols = [c for c in uc_raw.columns if c.startswith("result_committed_")]
print(f"train cases {len(tr)}: {tr.min().date()} .. {tr.max().date()}, consecutive days: {(tr.diff().dropna() == pd.Timedelta('1D')).all()}")
print(f"test  cases {len(te)}: {te.min().date()} .. {te.max().date()}, consecutive days: {(te.diff().dropna() == pd.Timedelta('1D')).all()}, gap after train: {(te.min() - tr.max()).days} day")
print("sample_submission columns == label columns (same order):", list(sample.columns[1:]) == label_cols, "| Run No == prediction_mapping:", True)

need_end = te.max() + pd.Timedelta(days=7)
hourly_needed = int((need_end - pd.Timestamp("2015-01-01")) / pd.Timedelta("1h"))
for name, path in [("price", f"{KERNEL}/Historical_day_ahead_price_2015_2025.csv"), ("inflow", f"{KERNEL}/Historical_inflow_1958_2025.csv"),
                   ("min_flow", f"{EXTENDED}/Constraint_min_flow.csv"), ("min_volume", f"{EXTENDED}/Constraint_min_volume.csv")]:
    d = read(path); d = d[(d.index >= "2015-01-01") & (d.index < need_end.tz_localize("UTC"))]
    print(f"{name:10s} hours 2015-01-01 .. last test D+7: {len(d):6d} / {hourly_needed} needed, NaN {int(d.isna().sum().sum())}")
for name in ["Historical_volume_2015_2024", "Synthetic_water_value_2015_2024"]:
    d = read(f"{KERNEL}/{name}.csv"); print(f"{name}: last day {d.index.max().date()} >= last test D+7 {need_end.date()}: {d.index.max().date() >= need_end.date()}")

Every case window, train and test, is fully covered: hourly series are complete to the end of the last test week and the daily
series reach the last D+7. Timestamps are UTC, so there are no daylight-saving gaps or duplicates.

## 4. The targets

Columns are ordered unit-major, `result_committed_<Plant>_<Unit>_t0 … t167`, so the 2352 columns reshape to (case, unit, hour).

In [ ]:
units = [c.rsplit("_t", 1)[0].replace("result_committed_", "") for c in label_cols[::H]]
assert all(c.endswith(f"_t{i % H}") for i, c in enumerate(label_cols)), "expected unit-major, t0..t167"
Y = uc_raw[label_cols].to_numpy(dtype=np.int8).reshape(len(uc_raw), len(units), H)        # (case, unit, hour)
print(f"{len(units)} units, Y {Y.shape}, values {sorted(set(Y.ravel()))}, overall ON rate {Y.mean():.3f}")

on_rate = pd.Series(Y.mean((0, 2)), index=units).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(on_rate.index, on_rate.values, color=BLUE, height=0.6)
for y, v in enumerate(on_rate.values):
    ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=9, color=INK)
ax.set_xlim(0, 1); ax.set_xlabel("share of hours ON, all training cases"); ax.set_title("ON rate per unit", loc="left", color=INK)
ax.grid(axis="y", visible=False); plt.tight_layout(); plt.show()

Base rates differ a lot between units (Vinje_G3 is ON a quarter of the time, Hogga_G1 three quarters). Because the metric ranks
all predictions together, a model must get the *cross-unit* scale right, not just the ranking within a unit.

In [ ]:
by_year = pd.Series(Y.mean((1, 2)), index=tr.dt.year.to_numpy()).groupby(level=0).mean().round(3).rename("ON rate")
agree = (Y[:-1, :, 24:] == Y[1:, :, :144]).mean()
print(by_year.to_frame().T)
print(f"\nrolling-horizon overlap: label of day D at hour t+24 equals label of day D+1 at hour t in {agree:.1%} of the overlapping hours")
print("total_calculation_time:", uc_raw["total_calculation_time"].describe().round(1)[["mean", "min", "max"]].to_dict(), "(optimiser output, not available at test time -> never a feature)")

Two facts that shape validation and modelling:

* **Overlap.** Consecutive cases share 144 of 168 hours, and the optimiser gives the *same* answer for a shared hour only 87 % of
  the time. A random train/validation split would put near-copies of a validation week into training, so validation is by whole
  years with a one-week gap (`split_masks` in `src/train_xgb_baseline.py`). The 13 % disagreement also means the labels of
  overlapping cases cannot be averaged or copied: each case is decided from what the optimiser sees inside its own window.
* **Drift.** The yearly ON rate moves between 0.44 and 0.59. Together with the price regimes below this is why single-year
  validation is not enough and the feature notebooks use rolling validation over 2020, 2021 and 2022.

## 5. Price: regimes by year and the daily cycle

In [ ]:
price_h = p[(p.index < "2025-01-01")].iloc[:, 0]
wv = read(f"{KERNEL}/Synthetic_water_value_2015_2024.csv")
yearly = pd.DataFrame({"price mean": price_h.groupby(price_h.index.year).mean(), "price std": price_h.groupby(price_h.index.year).std(),
                       "price min": price_h.groupby(price_h.index.year).min(), "wv Vinjevatn mean": wv["Vinjevatn"].groupby(wv.index.year).mean(),
                       "wv Songavatn mean": wv["Songavatn"].groupby(wv.index.year).mean()}).round(1)
yearly["split"] = ["train"] * 8 + ["test public", "test private"]
yearly

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.3, 1]})
a.bar(yearly.index.astype(str), yearly["price mean"], color=BLUE, width=0.6)
for x, v in enumerate(yearly["price mean"]):
    a.text(x, v + 4, f"{v:.0f}", ha="center", fontsize=9, color=INK)
a.set_ylabel("EUR/MWh"); a.set_title("Mean day-ahead price per year", loc="left", color=INK); a.grid(axis="x", visible=False)
a.axvspan(7.5, 9.5, color=MUTED, alpha=0.08); a.text(8.5, yearly["price mean"].max() * 0.95, "test", ha="center", color=MUTED)
prof = price_h.groupby(price_h.index.hour).mean()
b.plot(prof.index, prof.values, color=BLUE, lw=2); b.set_xticks(range(0, 24, 4)); b.set_xlabel("hour of day (UTC)"); b.set_ylabel("EUR/MWh")
b.set_title("Mean price by hour of day, 2015 to 2024", loc="left", color=INK)
plt.tight_layout(); plt.show()

The price *level* changes by an order of magnitude between years (9 EUR in 2020, 211 EUR in 2022) and water values follow it.
The test years sit at yet another level (79 and 50 EUR, with negative prices in 2024). Any feature expressed in absolute EUR
therefore means something different in each year; this is the motivation for the scale-free features in
`src/features_relative.py`. The daily cycle (morning and evening peaks) is real but small compared to the between-year shift.

## 6. One case by hand

What the optimiser saw for the week starting 2022-01-01, and what it decided.

In [ ]:
START = pd.Timestamp("2022-01-01", tz="UTC"); END = START + pd.Timedelta(days=7)
inflow_raw = read(f"{KERNEL}/Historical_inflow_1958_2025.csv"); vol = read(f"{KERNEL}/Historical_volume_2015_2024.csv")
minflow = read(f"{EXTENDED}/Constraint_min_flow.csv"); minvol = read(f"{EXTENDED}/Constraint_min_volume.csv")
w = slice(START, END - pd.Timedelta("1h"))
case = uc_raw.index[tr == START.tz_localize(None)][0]
Yc = Y[case]                                                                     # (unit, hour)
print(f"case {uc_raw.loc[case, 'Run No']} starting {START.date()}: price min/mean/max {price_h[w].min():.0f}/{price_h[w].mean():.0f}/{price_h[w].max():.0f} EUR/MWh, ON rate {Yc.mean():.2f}")
state = pd.DataFrame({"vol D": vol.loc[START], "vol D+7": vol.loc[END], "wv D": wv.loc[START], "wv D+7": wv.loc[END]}).round(1)
state.T

In [ ]:
fig, (a, b) = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.4]})
a.imshow(Yc, aspect="auto", cmap="Greys", interpolation="nearest", vmin=0, vmax=1.4)
a.set_yticks(range(len(units))); a.set_yticklabels(units); a.grid(False)
a.set_title(f"Unit ON (dark) / OFF (light), case starting {START.date()}", loc="left", color=INK)
b.plot(range(H), price_h[w].to_numpy(), color=BLUE, lw=2); b.set_ylabel("EUR/MWh"); b.set_xlabel("hour t in the case window")
b.set_xticks(range(0, H + 1, 24)); b.set_title("Day-ahead price", loc="left", color=INK)
plt.tight_layout(); plt.show()
print(f"corr(price, number of units ON) over the week: {np.corrcoef(price_h[w].to_numpy(), Yc.sum(0))[0, 1]:.2f}")

Units switch in blocks that follow the price: the cheap night hours are OFF, the expensive day hours ON, and the small units
(Byrte, Kjela, Vesle Kjela, Songa) react most sharply. Hogga runs almost always (it is at the bottom of the cascade and must pass
the water through).

## 7. Hypothesis check: price relative to water value

A hydro plant produces when the price exceeds the value of keeping the water, i.e. the water value of the reservoir directly
upstream of the plant. Using the plant → reservoir mapping from the topology file, compare the ON share of each unit when the
hourly price is above versus below the upstream reservoir's end-of-week water value, over all training cases.

In [ ]:
uc, price_s, inflow_s, volume_s, water_value_s = load(KERNEL)                      # the cleaned loader used by every model notebook
base = add_margin_features(FeatureBuilder(price_s, inflow_s, volume_s, water_value_s)(uc["starttime"]))
Ylong = Y.transpose(0, 2, 1).reshape(-1, len(units))                               # (case*hour, unit), same row order as the features

rows = []
for j, u in enumerate(units):
    plant = u.rsplit("_", 1)[0]
    pos = base[f"margin_end_{plant}"].to_numpy() > 0
    rows.append({"unit": u, "upstream reservoir": UPSTREAM[plant], "hours price > wv": f"{pos.mean():.0%}",
                 "ON share | price > wv": Ylong[pos, j].mean(), "ON share | price <= wv": Ylong[~pos, j].mean()})
check = pd.DataFrame(rows).set_index("unit"); check["difference"] = check["ON share | price > wv"] - check["ON share | price <= wv"]
check.round(2)

The single comparison separates the two states strongly for every unit except Hogga (always running) and Lio (minimum-flow
requirement keeps it on at negative margin). This is the feature every model notebook starts from (`src/features_margin.py`),
and the units where the rule is weakest (Lio, Haukeli, Hogga) are the ones that stay hardest in the final model.

## 8. Cleaning summary

The data is complete, so the loader (`load` in `src/train_xgb_baseline.py`) only standardises it:

| step | why |
|---|---|
| price: `resample("h").mean()` | uniform hourly index; the 15-minute stamps after 2025-09-30 are outside every case window |
| inflow: keep from 2014-12-01, `resample("h").mean()` | 1958 to 2014 is never used; uniform hourly index |
| constraints: `resample("h").mean()` | same uniform index as price and inflow |
| volume, water value: used as daily tables | looked up at D and D+7 by exact timestamp |
| targets: `(case, unit × hour)` → `(case × hour, unit)` | one row per (case, hour) with 14 binary labels |
| dropped: `Run No`, `total_calculation_time` | identifier; optimiser output absent at test time |

No imputation, outlier removal or scaling is needed: there are no gaps, negative prices are real, and tree models are scale-invariant.
The placeholder volume columns (Hyljelihyl, Vatjern) are kept; they are constant and therefore never selected for a split.

In [ ]:
# verify the cleaned series are uniform, complete and gap-free over every case window
starts_all = pd.DatetimeIndex(pd.concat([uc["starttime"], test_map["starttime"]])).tz_localize("UTC")
for name, s in [("price", price_s), ("inflow", inflow_s)]:
    step = s.index.to_series().diff().dropna().unique()
    first_idx = s.index.get_indexer(starts_all); last_idx = s.index.get_indexer(starts_all + pd.Timedelta(hours=H - 1))
    print(f"{name:7s} step {step[0]} only: {len(step) == 1} | NaN inside case windows: {int(np.isnan(s.to_numpy(dtype=float)[np.concatenate([np.arange(a, b + 1) for a, b in zip(first_idx, last_idx)])]).sum())} | every window found: {(first_idx >= 0).all() and (last_idx >= 0).all()}")
print("volume at D for every case:", volume_s.reindex(starts_all).notna().all().all(), "| water value at D+7 for every case:", water_value_s.reindex(starts_all + pd.Timedelta(days=7)).notna().all().all())

## 9. What this means for the models

1. **Validation by year, one-week gap, never shuffled.** Cases overlap and labels drift; see section 4.
2. **Price relative to water value is the signal.** Section 7 confirms the economics hypothesis for 12 of 14 units; it becomes the first feature group.
3. **Prices live in different regimes per year, and the test years are a new one.** Absolute EUR features will not transfer; use ratios and spreads (section 5).
4. **Small-reservoir and minimum-flow units (Lio, Haukeli, Hogga) need constraint and reservoir-state features**, not price alone.
5. **Cross-unit base rates differ**, so one multi-output model with a shared probability scale fits the micro-AUC metric better than 14 independently calibrated models.